In [1]:
# !pip install yfinance pandas FinMind tqdm

In [2]:
import yfinance as yf
import pandas as pd
from FinMind.data import DataLoader
import warnings
import os

# 忽略 pandas 的一些未來版本警告，保持終端機輸出乾淨
warnings.filterwarnings('ignore')

def fetch_predictive_modeling_data(start_date="2023-01-01", end_date="2024-01-01"):
    print("初始化資料下載模組...")
    
    # ==========================================
    # 1. 透過 yfinance 獲取價量、美股與總經資料
    # ==========================================
    # 包含您要研究的三檔台股標的，以及相關的跨國控制變數
    tickers = [
        "2330.TW", "2454.TW", "0050.TW", 
        "TSM", "^SOX", "^NDX", "^GSPC", "^VIX", 
        "TWD=X", "^TNX"
    ]
    
    print("=> 正在從 Yahoo Finance 下載全球市場資料...")
    yf_raw = yf.download(tickers, start=start_date, end=end_date)
    
    price_df = yf_raw['Close']
    volume_df = yf_raw['Volume']
    
    # 處理美股與台股休市日不同的問題 (使用新版 pandas 的 ffill() 函數)
    price_df.ffill(inplace=True)
    volume_df.ffill(inplace=True)


    # ==========================================
    # 2. 透過 FinMind 獲取台股微觀籌碼資料 (涵蓋 2330, 2454, 0050)
    # ==========================================
    print("=> 正在從 FinMind 獲取三大法人與籌碼資料...")
    api = DataLoader()
    
    # 欲抓取的台股目標清單
    target_stocks = ['2330', '2454', '0050']
    all_institutional = []
    all_margin = []
    
    for stock_id in target_stocks:
        print(f"   正在下載 {stock_id} 的法人與融資券資料...")
        
        # 取得三大法人買賣超
        inst_df = api.taiwan_stock_institutional_investors(
            stock_id=stock_id, 
            start_date=start_date, 
            end_date=end_date
        )
        if not inst_df.empty:
            all_institutional.append(inst_df)
            
        # 取得融資融券餘額 (0050 ETF 同樣具備融資券信用交易資料)
        mar_df = api.taiwan_stock_margin_purchase_short_sale(
            stock_id=stock_id, 
            start_date=start_date, 
            end_date=end_date
        )
        if not mar_df.empty:
            all_margin.append(mar_df)
            
    # 將多檔個股的 DataFrame 縱向合併
    institutional_df = pd.concat(all_institutional, ignore_index=True) if all_institutional else pd.DataFrame()
    margin_df = pd.concat(all_margin, ignore_index=True) if all_margin else pd.DataFrame()


    # ==========================================
    # 3. 透過 FinMind 獲取台指期貨夜盤資料
    # ==========================================
    print("=> 正在獲取台指期 (TX) 夜盤資料...")
    futures_df = api.taiwan_futures_daily(
        futures_id="TX", 
        start_date=start_date, 
        end_date=end_date
    )
    
    tx_night_df = pd.DataFrame()
    if not futures_df.empty:
        if 'trading_session' in futures_df.columns:
            # 1. 先篩選出夜盤資料
            night_mask = futures_df['trading_session'] == 'after_market'
            all_night_data = futures_df[night_mask]
            
            if not all_night_data.empty:
                # 2. 進階優化：在每天的夜盤中，只保留成交量最大（近月）的合約，排除遠月合約的干擾
                # 依據日期 (date) 分組，並找出每組中 volume 最大的索引
                idx = all_night_data.groupby('date')['volume'].idxmax()
                tx_night_df = all_night_data.loc[idx].sort_values('date')
                print(f"   -> 成功篩選出最具代表性的夜盤近月合約資料，共 {len(tx_night_df)} 筆。")
        else:
            print(f"   [警告] 找不到 'trading_session' 欄位。目前的欄位為: {list(futures_df.columns)}")
    print("資料下載與整合完成！\n")
    return price_df, volume_df, institutional_df, margin_df, tx_night_df


if __name__ == "__main__":
    # 設定您要研究的時間區間
    START_DATE = "2017-1-01"
    END_DATE = "2026-05-18"
    
    # 執行資料抓取 (回傳包含成交量 volumes)
    prices, volumes, inst_data, margin_data, tx_night = fetch_predictive_modeling_data(
        start_date=START_DATE, 
        end_date=END_DATE
    )

    # 建立一個專用資料夾，避免檔案散落
    output_dir = "taiwan_market_data"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("=> 正在將資料寫入 CSV 檔案...")
    
    # 1. 儲存全球價格資料 (yfinance 的 Index 是日期 Date，必須保留)
    prices.to_csv(f"{output_dir}/global_prices.csv", index=True)
    print(f"   [已儲存] 價格資料 -> {output_dir}/global_prices.csv")
    
    # 2. 儲存全球成交量資料 (選填，量能變化常作為基準模型的控制特徵)
    volumes.to_csv(f"{output_dir}/global_volumes.csv", index=True)
    print(f"   [已儲存] 成交量資料 -> {output_dir}/global_volumes.csv")
    
    # 3. 儲存三大法人買賣超資料 (FinMind 內建 date 欄位，不需保留預設的數字流水號 index)
    if not inst_data.empty:
        inst_data.to_csv(f"{output_dir}/institutional_investors.csv", index=False)
        print(f"   [已儲存] 三大法人籌碼 (含2330/2454/0050) -> {output_dir}/institutional_investors.csv")
        
    # 4. 儲存融資融券信用交易資料
    if not margin_data.empty:
        margin_data.to_csv(f"{output_dir}/margin_trading.csv", index=False)
        print(f"   [已儲存] 融資融券餘額 (含2330/2454/0050) -> {output_dir}/margin_trading.csv")
        
    # 5. 儲存台指期夜盤資料
    if not tx_night.empty:
        tx_night.to_csv(f"{output_dir}/tx_futures_night.csv", index=False)
        print(f"   [已儲存] 台指期夜盤數據 -> {output_dir}/tx_futures_night.csv")
    print(f"\n所有市場數據已成功儲存至 '{output_dir}/' 資料夾下。")

初始化資料下載模組...
=> 正在從 Yahoo Finance 下載全球市場資料...


[*********************100%***********************]  10 of 10 completed
2026-05-19 15:09:20.826 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-05-19 15:09:20.826 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2330


=> 正在從 FinMind 獲取三大法人與籌碼資料...
   正在下載 2330 的法人與融資券資料...


2026-05-19 15:09:21.172 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 2330
2026-05-19 15:09:21.356 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2454
2026-05-19 15:09:21.510 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 2454


   正在下載 2454 的法人與融資券資料...


2026-05-19 15:09:21.625 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 0050
2026-05-19 15:09:21.748 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 0050


   正在下載 0050 的法人與融資券資料...


2026-05-19 15:09:22.305 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanFuturesDaily, data_id: TX


=> 正在獲取台指期 (TX) 夜盤資料...
   -> 成功篩選出最具代表性的夜盤近月合約資料，共 2196 筆。
資料下載與整合完成！

=> 正在將資料寫入 CSV 檔案...
   [已儲存] 價格資料 -> taiwan_market_data/global_prices.csv
   [已儲存] 成交量資料 -> taiwan_market_data/global_volumes.csv
   [已儲存] 三大法人籌碼 (含2330/2454/0050) -> taiwan_market_data/institutional_investors.csv
   [已儲存] 融資融券餘額 (含2330/2454/0050) -> taiwan_market_data/margin_trading.csv
   [已儲存] 台指期夜盤數據 -> taiwan_market_data/tx_futures_night.csv

所有市場數據已成功儲存至 'taiwan_market_data/' 資料夾下。
